##### Cell 1 — Load the dataset and define features

In [1]:

# ==========================================================
# IT3091 - Shared Validation
# Responsibility: IT24101808
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

dataset_path = (
    project_root / "data" / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(dataset_path)

# Deterministic data quality correction
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

zero_tenure_mask = (
    df["tenure"].eq(0)
    & df["TotalCharges"].isna()
)
df.loc[zero_tenure_mask, "TotalCharges"] = 0.0

assert df["TotalCharges"].isna().sum() == 0
assert df["Churn"].isin(["Yes", "No"]).all()

# Separate target and predictors
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

print("Features:", X.shape)
print("Target:", y.shape)
print("Target values:", sorted(y.unique()))


Features: (7043, 19)
Target: (7043,)
Target values: [np.int64(0), np.int64(1)]


##### Cell 2 — Create the shared stratified split

In [2]:

# Responsibility: IT24101808 - Validation

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts())
print((y_test.value_counts(normalize=True) * 100).round(2))

print("\nOverlapping indices:",
      len(set(X_train.index) & set(X_test.index)))


Training features: (5634, 19)
Testing features: (1409, 19)

Training target distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    1035
1     374
Name: count, dtype: int64
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Overlapping indices: 0


##### Cell 3 — Define the shared cross-validation strategy

In [3]:

# Responsibility: IT24101808 - Validation

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train),
    start=1
):
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    fold_results.append({
        "Fold": fold,
        "Train Samples": len(train_idx),
        "Validation Samples": len(val_idx),
        "Train Churn %": round(
            y_fold_train.mean() * 100, 2
        ),
        "Validation Churn %": round(
            y_fold_val.mean() * 100, 2
        )
    })

display(pd.DataFrame(fold_results))


,Fold,Train Samples,Validation Samples,Train Churn %,Validation Churn %
0,1,4507,1127,26.54,26.53
1,2,4507,1127,26.54,26.53
2,3,4507,1127,26.54,26.53
3,4,4507,1127,26.54,26.53
4,5,4508,1126,26.53,26.55


##### Verify the shared foundation

In [4]:

# ==========================================================
# Verify shared project functions
# ==========================================================

import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_preparation import (
    load_and_clean_data,
    separate_features_target
)

from src.validation import (
    create_train_test_split,
    get_cross_validation,
    RANDOM_STATE,
    PRIMARY_METRIC
)

shared_df = load_and_clean_data(project_root)

shared_X, shared_y = separate_features_target(shared_df)

(
    shared_X_train,
    shared_X_test,
    shared_y_train,
    shared_y_test
) = create_train_test_split(shared_X, shared_y)

shared_cv = get_cross_validation()

print("SHARED FOUNDATION VERIFICATION")
print("-" * 45)

print("Dataset shape:", shared_df.shape)
print("Features:", shared_X.shape)
print("Training set:", shared_X_train.shape)
print("Testing set:", shared_X_test.shape)

print("Random state:", RANDOM_STATE)
print("CV folds:", shared_cv.n_splits)
print("Primary metric:", PRIMARY_METRIC)

print("Missing values:", shared_df.isna().sum().sum())

print(
    "Training churn rate:",
    round(shared_y_train.mean() * 100, 2)
)

print(
    "Testing churn rate:",
    round(shared_y_test.mean() * 100, 2)
)

print(
    "Train/test overlap:",
    len(
        set(shared_X_train.index)
        & set(shared_X_test.index)
    )
)


SHARED FOUNDATION VERIFICATION
---------------------------------------------
Dataset shape: (7043, 21)
Features: (7043, 19)
Training set: (5634, 19)
Testing set: (1409, 19)
Random state: 42
CV folds: 5
Primary metric: f1
Missing values: 0
Training churn rate: 26.54
Testing churn rate: 26.54
Train/test overlap: 0
